# 08_Turnover_Cost_Analysis — 換倉週轉率與交易成本分析

## 目的

Phase 3-E 核心研究：檢視策略的「換股頻率 (Turnover Rate)」。

如果我們每個月的選股名單都 100% 不一樣（大換血），代表我們每個月的資金都要被扣除 0.1425% (手續費) + 0.3% (交易稅) + 0.1% (滑價) 的高昂成本。長久下來，再高的 CAGR 也會被國家跟券商吃光。

這個實驗室會分析目前 `TOP_K=40` 的情況下，我們平均每個月換了多少比例的股票？每年我們到底繳了多少成本？

In [6]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. 切換到專案根目錄讀取資料
original_cwd = os.getcwd()
try:
    if os.path.basename(original_cwd) == "Research":
        os.chdir("..")
        
    print("Loading strategy weights...\n")
    weights_df = pd.read_pickle("weights.pkl")
finally:
    os.chdir(original_cwd)
    
# 2. 計算 Turnover (換倉率)
# Turnover = 月對月權重變化絕對值總和 / 2
# 0% 代表都沒換，100% 代表全部賣掉買新的

# 確保 index 為 datetime 且只看有持股的月份
weights_df.index = pd.to_datetime(weights_df.index)
active_months = weights_df[(weights_df > 0).sum(axis=1) > 0]

# 計算月對月的差值
weight_diff = active_months.diff().fillna(0)
turnover = weight_diff.abs().sum(axis=1) / 2

# 第一個月建倉算 100% 買入，這裡我們忽略第一個月，只看後續的日常換倉
turnover = turnover.iloc[1:]

avg_turnover = turnover.mean()
max_turnover = turnover.max()

print(f"=== Turnover Analysis ===")
print(f"平均月換倉率: {avg_turnover:.2%}")
print(f"最高月換倉率: {max_turnover:.2%}")
print(f"一年平均換倉: {avg_turnover * 12:.2f} 次 (代表整個資金池一年被翻攪幾次)")

# 3. 計算隱形交易成本
# 單次換倉成本 = 賣出稅(0.3%) + 賣出手續費(0.1425%) + 買入手續費(0.1425%) + 雙向滑價(0.1%*2)
# 粗估來回一趟總摩擦成本約 0.785% (0.00785)
FRICTION_COST_PER_TURN = 0.003 + 0.001425 + 0.001425 + 0.002

monthly_cost = turnover * FRICTION_COST_PER_TURN
annual_cost = avg_turnover * 12 * FRICTION_COST_PER_TURN
total_accumulated_cost = monthly_cost.sum()

print(f"\n=== Transaction Cost Analysis ===")
print(f"來回換倉摩擦成本率: {FRICTION_COST_PER_TURN:.2%}")
print(f"每年估計被吃掉的報酬率: {annual_cost:.2%}")
print(f"回測全期({len(turnover)}個月)累計被吃掉的報酬率: {total_accumulated_cost:.2%}")


Loading strategy weights...

=== Turnover Analysis ===
平均月換倉率: 74.13%
最高月換倉率: 97.50%
一年平均換倉: 8.90 次 (代表整個資金池一年被翻攪幾次)

=== Transaction Cost Analysis ===
來回換倉摩擦成本率: 0.78%
每年估計被吃掉的報酬率: 6.98%
回測全期(60個月)累計被吃掉的報酬率: 34.92%


In [7]:
# 4. 視覺化換倉率與成本
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=turnover.index, 
    y=turnover * 100, 
    name="Monthly Turnover (%)",
    marker_color="#FF9800"
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=monthly_cost.index, 
    y=monthly_cost.cumsum() * 100,
    name="Cumulative Cost (%)",
    mode="lines",
    line=dict(color="#F44336", width=3),
    fill='tozeroy'
), secondary_y=True)

fig.add_hline(y=avg_turnover * 100, line_dash="dash", line_color="#FFEB3B", 
              annotation_text=f"Avg: {avg_turnover:.1%}", annotation_position="top right")

fig.update_layout(
    title="Monthly Portfolio Turnover & Cumulative Friction Cost",
    template="plotly_dark", height=500,
    legend=dict(x=0.01, y=0.99)
)
fig.update_yaxes(title_text="Turnover Rate (%)", secondary_y=False, range=[0, 100])
fig.update_yaxes(title_text="Cumulative Cost (%)", secondary_y=True)

fig.show()

## 結論與下一步方案

如果「平均月換倉率」超過 50%：
代表我們每個月有一半的股票都在被汰換，每年資金池被洗了超過 6 次，每年會吃掉超過 4.7% 的淨利。這會導致我們的策略看起來很會賺，但實際上都是在幫券商打工。

**潛在的優化方向 (Turnover Constraint)：**
1. **惰性換倉 (Inertia) / 留校察看機制**：在 `strategy.py` 選股時，如果目前手上有這支股票，且它的 AI 評分雖然掉出前 40 名，但還維持在「前 60 名」以內，我們就不要賣它！只有當它真的爛到掉出前 60 名，我們才忍痛賣出付手續費。
2. **單月最大換股數量限制**：強制規定每個月最多只能更換 X 檔股票（不過這可能會讓模型無法及時應對崩盤股）。

不過這會增加程式碼複雜度，而且 `vectorbt` 已經在剛才的 8.55% 裡「誠實扣除」了這些費用。如果換出來的獲利大於摩擦成本，那高換倉率也是可以接受的。